In [ ]:
!pip install -r requirements.txt

In [ ]:
import pandas as pd
import pm4py

# Modellierung mit SNAKES

In [ ]:
from snakes.nets import *
import snakes.plugins
snakes.plugins.load("gv", "snakes.nets", "nets")
from nets import *

# Erstellen eines Petri-Netzes mit dem Namen 'First net'
n = PetriNet('First net')
# Hinzufügen einer Stelle mit dem Namen 'p' und Anfangsmarkierung '0'
n.add_place(Place('p', [0]))
#Hinzufügen einer Transition mit dem Namen 't' und der Bedingung 'x<5'
n.add_transition(Transition('t', Expression('x<5')))
# Hinzufügen einer Eingangskante von Stelle 'p' zur Transition 't' mit der Variable 'x'
n.add_input('p', 't', Variable('x'))
# Hinzufügen einer Ausgangskante von der Transition 't' zur Stelle 'p' mit dem Ausdruck 'x+1'
n.add_output('p', 't', Expression('x+1'))

# Graphische Abbildung des Petri-Netzes
n.draw("First_net.png")

In [ ]:
import snakes.plugins
snakes.plugins.load("gv", "snakes.nets", "nets")
from nets import *

def factory (cons, prod, init=[1, 2, 3]):
    """
    Erstelle ein Petri-Netz mit einer Stelle 'src', einer Stelle 'snk', 
    einer Transition 't' und den zugehörigen Kanten.

    Args:
        cons -- Bedingung für die Eingangskante von 'src' zu 't'
        prod -- Ausdruck für die Ausgangskante von 't' zu 'snk'
        init -- Anfangsmarkierung von 'src' (Default: [1,2,3])

    Returns:
        n -- Das erstellt Petri-Netz
        t -- Die erstellte Transition 't'
        modes -- Eine Liste der Transitionemodi von 't'
    """
    # Erstellen eines Petri-Netzes mit dem Namen 'N'
    n = PetriNet("N")
    # Hinzufügen einer Stelle mit dem Namen 'src' und der Anfangsmarkierung 'init'
    n.add_place(Place("src", init))
    # Hinzufügen einer Stelle mit dem Namen 'snk' und einer leeren Markierung
    n.add_place(Place("snk", []))
    # Erstellen einer Transition mit dem Namen 't'
    t = Transition("t")
    # Hinzufügen der Transition zum Petri-Netz
    n.add_transition(t)
    # Hinzufügen einer Eingangskante von der Stelle 'src' zur Transition 't' mit der Bedingung 'cons'
    n.add_input("src", "t", cons)
    # Hinzufügen einer Ausgangskante von der Transition 't' zur Stelle 'snk' mit dem Ausdruck 'prod'
    n.add_output("snk", "t", prod)
    # Rückgabe des Petri-Netzes, der Transition 't' und der Liste der Transitionenmodi von 't'
    return n, t, t.modes()

In [ ]:
# Aufruf der Funktion 'factory' mit den Parametern 'Value(1)' und 'Value(0)'
# Das Ergebnis wird den Variablen 'net', 'trans' und 'modes' zugewiesen
net, trans, modes = factory(Value(1), Value(0))
# Zeichnen des Petri-Netzes 'net' und Speichern als Bild mit dem Dateinamen "value-0.png"
net.draw("value-0.png")
# Ausgabe der Liste 'modes'
print(modes)
# Schalten der Transition 'trans' mit dem Modus aus der ersten Position der Liste 'modes'
trans.fire(modes[0])
# Zeichnen des Petri-Netzes 'net' nach dem Schalten der Transition und Speichern als Bild mit dem Dateinamen "value-1.png"
net.draw("value-1.png")

# Soundness-Check mit PM4Py

In [ ]:
# Einlesen der CSV-Datei "./KKR.csv" mit Trennzeichen ";"
log = pd.read_csv("./KKR.csv", sep=";")

# Konvertieren der Spalte 'timestamp' in das Datumsformat 'dd.mm.yyyy hh:mm'
log['timestamp'] = pd.to_datetime(log['timestamp'], format='%d.%m.%Y %H:%M')
# Konvertieren der Spalte 'activity' in den Datentyp 'str'
log['activity'] = log['activity'].astype(str)
# Konvertieren der Spalte 'case' in den Datentyp 'str'
log['case'] = log['case'].astype(str)

# Umbenennen der Spalten 'case' in 'case:concept:name', 'activity' in 'concept:name' und 'timestamp' in 'time:timestamp'
log = log.rename(columns={'case': 'case:concept:name', 'activity': 'concept:name', 'timestamp': 'time:timestamp'})

In [ ]:
net, initial_marking, final_marking = pm4py.discovery.discover_petri_net_alpha(log)

pm4py.vis.view_petri_net(net, initial_marking, final_marking)

In [ ]:
wfn = pm4py.analysis.check_is_workflow_net(net)

if wfn == True:
    print('Das Petri-Netz ist ein Workflow-Netz.')
else:
    print('Das Petri-Netz ist kein Workflow-Netz.')

In [ ]:
sound = pm4py.analysis.check_soundness(net, initial_marking, final_marking)

if sound[0] == True:
    print('\n -> Das Workflow-Netz ist sound.')
else: 
    print('\n -> Das Workflow-Netz ist nicht sound.')